# Cross-cohort inference, Jackson → Danenberg: one set of predictions, three metrics

**Purpose:** the data for the main figure. Every model is trained on Jackson and scored on the 130 frozen Danenberg test patients. Each model is scored three ways:
- **(a) pooled pixel r**, as in ImmuVis: one Pearson r per marker over all pixels of all test images;
- **(b) per-image pixel r**, as in VirTues: r within each image, then the mean over images;
- **(c) R4**: per-cell r after removing each image's mean.

**Models:**
1. ResNet-UNet (existing `kill-test` predictions, all 3 preprocessing arms; ImmuVis is the main one);
2. Table (LightGBM), trained on Jackson;
3. MLP, trained on Jackson;
4. GCN (Kipf) and GraphSAGE, trained on Jackson;
5. Table trained on the Danenberg train patients: the within-cohort reference.
6. **Reference row (not in the figure):** painted true cell means, the best any per-cell map can do on (a) and (b).
7. **Normalisation control:** the same Jackson-trained per-cell models applied to Danenberg inputs scaled with **Jackson** statistics
   (`[Jackson-scaled inputs]`), instead of Danenberg's own median and p5–p95 range. Like the image models, they then get no cohort-level
   adaptation to Danenberg.

**Per-cell models in pixel space:**
- Each cell's prediction fills its mask pixels, and the background is 0.
- Everything is scored in the image model's target space: `clip((arcsinh(x/5) − p01)/(p99 − p01), 0, 1)` with Jackson training percentiles.
- **Jackson-trained predictions** are mapped back with Jackson-only statistics (level A): first the inverse harmonisation, then the same transform.
- **The Danenberg-trained Table** learns the target-space cell means directly.
- **Pixel r follows exactly from per-image sums:**
  - n, Σg, Σgg come from `kill-test`'s `pixel_sums_per_image.csv`;
  - Σp, Σpp, Σgp come from cell pixel counts × cell values.
- A direct pixel check on 2 images verifies this.

**Shared with `train-test-grid`:** the code, the replicate-0 split and the seeds, so R4 should reproduce that run.

**Uncertainty:** 1000× patient bootstrap. The same draws are used for every metric and marker, so the median over the 10 markers gets a CI. ERα is excluded because it saturates in target space.

In [ ]:
import os, glob, json, time, sys, subprocess, warnings, traceback, re, pickle, gc, random
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from scipy.spatial import cKDTree
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LinearRegression
from joblib import Parallel, delayed
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=True)
    import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1"
SEED = 20261013
N_BOOT = 100 if SMOKE else 1000
N_BOOT_F1 = 50 if SMOKE else 200
N_SHIFT = 5
MIN_CELLS = 200
COF = 5.0
RING_R = 30.0
N_FIT = 15_000 if SMOKE else 100_000          # cells per cross-fit half (R5) and per classifier
N_FIT_FINAL = 30_000 if SMOKE else 300_000    # Jackson cells for the baselines
LGB = dict(n_estimators=40 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=os.cpu_count(), verbose=-1)
ARMS = ["residual", "gamma", "immuvis"]
NUC = ["histone_h3", "dna1", "dna2"]
# Image channel orders (as in AbundanceVsStructure). Danenberg index 12 'c_erb_b_2_her2' is clone 3B5.
D_CHAN = ['histone_h3', 'sma', 'cytokeratin_5', 'cd38', 'hla_dr', 'cytokeratin_8_18', 'cd15', 'fsp1', 'cd163',
          'cd278_icos', 'cd134', 'cd68', 'c_erb_b_2_her2', 'cd3', 'podoplanin', 'cd11c', 'cd279_pd_1', 'gitr_tnfrsf18',
          'cd16', 'c_erb_b_2_her2_d8f12', 'cd45ra', 'beta_2_microglobulin', 'cd45', 'foxp3', 'cd20',
          'estrogen_receptor_alpha', 'cd8a', 'cd57', 'ki_67', 'cd140b_pdgf_receptor_beta', 'caveolin_1', 'cd4',
          'cd31_v_wf', 'cxcl12_sdf_1', 'hla_abc', 'pan_cytokeratin', 'cleaved_caspase3', 'dna1', 'dna2']
J_CHAN = ['histone_h3', 'histone_h3_trimethylate', 'cytokeratin_5', 'fibronectin', 'cytokeratin_19', 'cytokeratin_8_18',
          'twist', 'cd68', 'keratin_14_krt14', 'sma', 'vimentin', 'c_myc', 'c_erb_b_2_her2', 'cd3', 'histone_h3_phospho',
          'slug', 'estrogen_receptor_alpha', 'progesterone_receptor_a_b', 'p53', 'cd44', 'cd45', 'gata3', 'cd20',
          'carbonic_anhydrase_ix', 'e_cadherin_p_cadherin', 'ki_67', 'egfr', 's6', 'v_wf', 'm_tor', 'cytokeratin_7',
          'pan_cytokeratin', 'cleaved_parp', 'dna1', 'dna2']
D_TABLE = {"c_erb_b_2_her2": "c_erb_b_2_her2_3b5"}       # image name -> Danenberg cell-table column
MARKERS = sorted(m for m in set(D_CHAN) & set(J_CHAN) if m not in NUC)    # = task_names in the training code
assert len(MARKERS) == 11, MARKERS
STROMAL_KEYS = ("fibro", "endothel", "stroma", "pdpn", "myofib", "vessel", "vwf")
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

def guard(name):
    """Run an analysis block; on failure print the traceback and carry on, so the kernel ends COMPLETE."""
    def deco(fn):
        try:
            return fn()
        except Exception:
            print(f"!!! block {name} failed:\n{traceback.format_exc()}", flush=True)
            FAILED.append(name)
            return None
    return deco
FAILED = []

## Cell tables (Danenberg cells to analyse; Jackson cells for the baselines)

In [ ]:
P = {}
for p in g("cells.parquet"):
    key = "danenberg" if "metabric_id" in pq.read_schema(p).names else "jackson"
    P[key] = dict(cells=p, contacts=near("cell_neighbours.parquet", p), report=near("prep_report.json", p))
P["danenberg"].update(clinical=near("clinical.parquet", P["danenberg"]["cells"]),
                      split=near("danenberg_patient_split.csv", P["danenberg"]["cells"]))
print(json.dumps(P, indent=1))

def load_table(key):
    cells = pd.read_parquet(P[key]["cells"])
    if key == "danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid = "metabric_id"
    else:
        cells = cells[cells.is_tumour == 1]
        pid = "patient_id"
    n_per = cells.groupby("image_name").size()
    cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    img_code, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
    loc = pd.Series(np.arange(len(cells)), index=pd.MultiIndex.from_arrays(
        [cells.image_number.values, cells.object_number.values]))
    con = pd.read_parquet(P[key]["contacts"])
    ca = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["from"].values])).values
    cb = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["to"].values])).values
    ok = ~(np.isnan(ca) | np.isnan(cb))
    ca, cb = ca[ok].astype(np.int64), cb[ok].astype(np.int64)
    ca, cb = ca[ca != cb], cb[ca != cb]
    if key == "danenberg":
        epi_frac = cells.groupby("cell_phenotype").is_epithelial.mean()
        bmap = {p: "epithelial" if epi_frac[p] > 0.5 else
                "stromal" if any(s in p.lower() for s in STROMAL_KEYS) else "immune" for p in epi_frac.index}
        broad = cells.cell_phenotype.map(bmap).astype(str).values
        epi = cells.is_epithelial.values == 1
    else:
        broad = cells.broad_class.astype(str).values
        epi = broad == "epithelial"
    return dict(key=key, cells=cells, img=img_code, img_names=np.asarray(img_names), starts=starts,
                patient=cells[pid].astype(str).values, xy=cells[["x", "y"]].values.astype(np.float64),
                ca=ca, cb=cb, broad=broad, epi=epi, phen=cells.cell_phenotype.astype(str).values,
                log_area=np.log(np.maximum(cells["area"].values, 1)).astype(np.float32))

DD, DJ = load_table("danenberg"), load_table("jackson")
N = len(DD["cells"])
split = pd.read_csv(P["danenberg"]["split"]).set_index("metabric_id").split
DD["split"] = pd.Series(DD["patient"]).map(split).fillna("none").values
clin = pd.read_parquet(P["danenberg"]["clinical"]).replace({"nan": np.nan})
clin = clin.assign(pid=clin.metabric_id.astype(str)).groupby("pid").first()
print(f"Danenberg: {N:,} cells, {len(DD['img_names'])} images, {len(set(DD['patient']))} patients; "
      f"split cells {pd.Series(DD['split']).value_counts().to_dict()}")
print(f"Jackson: {len(DJ['cells']):,} cells, {len(DJ['img_names'])} images")
pats, pcode = np.unique(DD["patient"], return_inverse=True)
W = np.random.default_rng(SEED).multinomial(len(pats), np.full(len(pats), 1 / len(pats)), size=N_BOOT).astype(np.float64)
tick("tables loaded")

def contact_matrix(D):
    n = len(D["cells"])
    C = sp.coo_matrix((np.ones(len(D["ca"]), np.float32), (D["ca"], D["cb"])), shape=(n, n)).tocsr()
    C = (C + C.T).tocsr()
    C.data[:] = 1.0
    return C

def ring_means(X, D, C):
    """Mean of X over non-touching cells within RING_R of the centroid, and their count."""
    n, p = X.shape
    Mr = np.full((n, p), np.nan, np.float32)
    nr = np.zeros(n, np.float32)
    for a, b in zip(D["starts"][:-1], D["starts"][1:]):
        m = b - a
        pr = cKDTree(D["xy"][a:b]).query_pairs(RING_R, output_type="ndarray")
        R = sp.coo_matrix((np.ones(len(pr), np.float32), (pr[:, 0], pr[:, 1])), shape=(m, m)).tocsr()
        R = (R + R.T).tocsr()
        Ci = C[a:b, a:b].tocsr()
        R = (R - R.multiply(Ci)).tocsr()
        R.eliminate_zeros()
        dr = np.asarray(R.sum(1)).ravel()
        Mr[a:b] = np.where(dr[:, None] > 0, (R @ X[a:b]) / np.maximum(dr, 1)[:, None], np.nan)
        nr[a:b] = dr
    return Mr, nr

## Reuse `kill-test` per-cell truth and ResNet-UNet predictions; recompute Danenberg per-cell image means

In [ ]:
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "imagecodecs==2024.9.22"]).returncode:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "imagecodecs"], check=True)   # 2024.9.22 no longer offered
import tifffile
KT = pd.read_parquet(g("percell_truth_pred.parquet")[0])
assert (KT.image_name.values == DD["cells"].image_name.values).all() and \
       (KT.object_number.values == DD["cells"].object_number.values).all(), "kill-test rows do not match the cell table"
TRUTH = {m: KT[f"truth__{m}"].values.astype(np.float32) for m in MARKERS}
RESNET = {(a, m): KT[f"pred__{a}__{m}"].values.astype(np.float32) for a in ARMS for m in MARKERS}
del KT

tifs = g("MB_*.tiff")
D_IMG = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if "/masks/" not in p.replace("\\", "/")}
D_MASK = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs if "/masks/" in p.replace("\\", "/")}
IMGS = list(DD["img_names"])
OBJ = DD["cells"].object_number.values.astype(np.int64)
ST = DD["starts"]

def cell_mean(flat, pm):
    v, loc, n, _ = pm
    cnt = np.bincount(loc, minlength=n)
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(cnt > 0, np.bincount(loc, flat[v], n) / np.maximum(cnt, 1), np.nan).astype(np.float32)

def raw_means(img_path, mask_path, obj):
    m = tifffile.imread(mask_path).astype(np.int64)
    lut = np.full(max(int(m.max()), int(obj.max())) + 1, -1, np.int64)
    lut[obj] = np.arange(len(obj))
    loc = lut[m.ravel()]
    v = np.flatnonzero(loc >= 0)
    pm = (v, loc[v], len(obj), m.shape)
    im = tifffile.imread(img_path)
    if im.ndim == 3 and im.shape[0] == len(D_CHAN) and im.shape[-1] != len(D_CHAN):
        im = np.moveaxis(im, 0, -1)
    im = im.reshape(-1, im.shape[-1]).astype(np.float32)
    return np.stack([cell_mean(im[:, c], pm) for c in range(im.shape[1])], 1)

RAW = np.vstack(Parallel(n_jobs=os.cpu_count())(delayed(raw_means)(D_IMG[s], D_MASK[s], OBJ[a:b])
                                                for s, a, b in zip(IMGS, ST[:-1], ST[1:])))
assert RAW.shape == (N, len(D_CHAN))
tick("kill-test outputs loaded; per-cell image means recomputed")

## Features and graphs (both cohorts)

In [ ]:
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
    import torch
import torch.nn as nn
import torch.nn.functional as F
dev = "cuda" if torch.cuda.is_available() else "cpu"
K_NN = 10
EPOCHS = 30 if SMOKE else 400
PATIENCE = 10 if SMOKE else 40
HID = 32 if SMOKE else 128
BUDGET_MIN = float(os.environ.get("BUDGET_MIN", 660))
REPS = [int(r) for r in os.environ.get("REPS", "0,1,2").split(",")]
SOURCES = ["Jackson", "Danenberg", "Both"]
MODELS = ["Table", "MLP", "GCN", "GraphSAGE"]
OTHER = {"Jackson": "Danenberg", "Danenberg": "Jackson"}

def harm(X):
    med = np.nanmedian(X, 0)
    q05, q95 = np.nanpercentile(X, [5, 95], 0)
    sc = np.where(q95 - q05 > 1e-3, q95 - q05, np.nanstd(X, 0) + 1e-6)
    return ((X - med) / sc).astype(np.float32)

dix = {m: i for i, m in enumerate(D_CHAN)}
IN_SET = NUC + MARKERS
XD_in = np.arcsinh(RAW[:, [dix[c] for c in IN_SET]] / COF).astype(np.float32)
XJ_in = np.arcsinh(DJ["cells"][IN_SET].values.astype(np.float64) / COF).astype(np.float32)
H = {"Jackson": np.c_[harm(XJ_in), harm(DJ["log_area"][:, None])],
     "Danenberg": np.c_[harm(XD_in), harm(DD["log_area"][:, None])]}
nJ, nD = len(H["Jackson"]), len(H["Danenberg"])
print(f"non-finite targets: Jackson {int((~np.isfinite(XJ_in)).sum())}, Danenberg {int((~np.isfinite(XD_in)).sum())}")

def knn_adj(D):
    n = len(D["cells"])
    rows, cols = [], []
    for a, b in zip(D["starts"][:-1], D["starts"][1:]):
        k = min(K_NN, b - a - 1)
        _, nb = cKDTree(D["xy"][a:b]).query(D["xy"][a:b], k=k + 1)
        rows.append(np.repeat(np.arange(a, b), k))
        cols.append(nb[:, 1:].ravel() + a)
    A = sp.coo_matrix((np.ones(sum(map(len, rows)), np.float32), (np.concatenate(rows), np.concatenate(cols))),
                      shape=(n, n)).tocsr()
    return ((A + A.T) > 0).astype(np.float32).tocsr()

def normalise(A):
    n = A.shape[0]
    deg = np.asarray(A.sum(1)).ravel()
    A_mean = (sp.diags(1 / np.maximum(deg, 1)) @ A).tocsr()                # GraphSAGE mean over neighbours
    Ai = A + sp.eye(n, dtype=np.float32, format="csr")
    dh = 1 / np.sqrt(np.asarray(Ai.sum(1)).ravel())
    A_gcn = (sp.diags(dh) @ Ai @ sp.diags(dh)).tocsr()                     # Kipf normalisation with self-loops
    return {"mean": A_mean, "gcn": A_gcn}, float(deg.mean())

ADJ = {}
ADJ["Jackson"], degJ = normalise(knn_adj(DJ))
ADJ["Danenberg"], degD = normalise(knn_adj(DD))
print(f"kNN graphs: mean degree Jackson {degJ:.1f}, Danenberg {degD:.1f}; device {dev}")

def to_csr(M):
    M = M.tocsr().astype(np.float32)
    M.sort_indices()
    return torch.sparse_csr_tensor(torch.from_numpy(M.indptr.astype(np.int32)), torch.from_numpy(M.indices.astype(np.int32)),
                                   torch.from_numpy(M.data), size=M.shape).to(dev)

class _SpMM(torch.autograd.Function):
    """A @ X with a precomputed CSR transpose for the backward pass."""
    @staticmethod
    def forward(ctx, X, A, AT):
        ctx.AT = AT
        return A @ X
    @staticmethod
    def backward(ctx, G):
        return ctx.AT @ G.contiguous(), None, None

def spmm(Ab, X):
    return _SpMM.apply(X, Ab[0], Ab[1])

def build_graph(src):
    """GPU tensors for a training/prediction graph: features, A (and A^T), row sums, and A @ features."""
    if src == "Both":
        mats = {k: sp.block_diag([ADJ["Jackson"][k], ADJ["Danenberg"][k]], format="csr") for k in ("mean", "gcn")}
        Xn = np.r_[H["Jackson"], H["Danenberg"]]
    else:
        mats, Xn = ADJ[src], H[src]
    Xn = np.nan_to_num(Xn)                                                 # NaN -> 0, the cohort median
    gr = {"x": torch.from_numpy(Xn).to(dev)}
    for k, M in mats.items():
        A = to_csr(M)
        gr[k] = (A, A if k == "gcn" else to_csr(M.T))
        gr["a1_" + k] = torch.from_numpy(np.asarray(M.sum(1)).ravel().astype(np.float32)).to(dev)
        gr["ax_" + k] = torch.from_numpy((M @ Xn).astype(np.float32)).to(dev)
    return gr

GR = {}
def ensure(keys):
    """Keep only the graphs in keys on the GPU (the pooled graph is large)."""
    for k in [k for k in GR if k not in keys]:
        del GR[k]
    if dev == "cuda":
        torch.cuda.empty_cache()
    for k in keys:
        if k not in GR:
            GR[k] = build_graph(k)

ensure({"Jackson", "Danenberg"})
tick("features and graphs")

## Models (identical maths to `model-comparison`; first-layer aggregation precomputed)

In [ ]:
from torch.utils.checkpoint import checkpoint
CKPT = {"on": False}       # gradient checkpointing (memory only; same maths), used for the pooled graph

def run(fn, *a):
    return checkpoint(fn, *a, use_reentrant=False) if (CKPT["on"] and torch.is_grad_enabled()) else fn(*a)

class MLP(nn.Module):
    def __init__(s, d, h=HID, p=0.1):
        super().__init__()
        s.net = nn.Sequential(nn.Linear(d, h), nn.ReLU(), nn.Dropout(p), nn.Linear(h, h), nn.ReLU(), nn.Dropout(p),
                              nn.Linear(h, 1))
    def forward(s, x, ax, A, a1):
        return s.net(x).squeeze(-1)

class GCN(nn.Module):
    def __init__(s, d, h=HID, p=0.1):
        super().__init__()
        s.l1, s.l2, s.out, s.dp = nn.Linear(d, h), nn.Linear(h, h), nn.Linear(h, 1), nn.Dropout(p)
    def forward(s, x, ax, A, a1):
        # layer 1: A (x W1 + b1) = (A x) W1 + (A 1) b1, as one product
        l1 = lambda ax, a1: s.dp(torch.relu(F.linear(torch.cat([ax, a1[:, None]], 1),
                                                     torch.cat([s.l1.weight, s.l1.bias[:, None]], 1))))
        l2 = lambda h: torch.relu(spmm(A, s.l2(h)))
        return s.out(run(l2, run(l1, ax, a1))).squeeze(-1)

class SAGE(nn.Module):
    def __init__(s, d, h=HID, p=0.1):
        super().__init__()
        s.s1, s.n1 = nn.Linear(d, h), nn.Linear(d, h, bias=False)
        s.s2, s.n2 = nn.Linear(h, h), nn.Linear(h, h, bias=False)
        s.out, s.dp = nn.Linear(h, 1), nn.Dropout(p)
    def forward(s, x, ax, A, a1):
        # layer 1: x S1 + A (x N1) = [x, A x] [S1; N1], as one product
        l1 = lambda x, ax: s.dp(torch.relu(F.linear(torch.cat([x, ax], 1), torch.cat([s.s1.weight, s.n1.weight], 1),
                                                    s.s1.bias)))
        l2 = lambda h: torch.relu(s.s2(h) + spmm(A, s.n2(h)))
        return s.out(run(l2, run(l1, x, ax))).squeeze(-1)

NETS = {"MLP": (MLP, "mean"), "GCN": (GCN, "gcn"), "GraphSAGE": (SAGE, "mean")}

def net_inputs(gr, akey, cols):
    return gr["x"][:, cols], gr["ax_" + akey][:, cols], gr[akey], gr["a1_" + akey]

def train_net(cls, inp, y, tr, va, seed):
    torch.manual_seed(seed)
    m = cls(inp[0].shape[1]).to(dev)
    opt = torch.optim.Adam(m.parameters(), lr=3e-3, weight_decay=1e-5)
    tr, va = torch.from_numpy(tr).to(dev), torch.from_numpy(va).to(dev)
    best, state, bad, ep = np.inf, None, 0, 0
    for ep in range(EPOCHS):
        m.train()
        opt.zero_grad()
        loss = ((m(*inp)[tr] - y[tr]) ** 2).mean()
        loss.backward()
        opt.step()
        if ep % 5 == 4:
            m.eval()
            with torch.no_grad():
                v = ((m(*inp)[va] - y[va]) ** 2).mean().item()
            if v < best - 1e-5:
                best, bad = v, 0
                state = {k: t.detach().clone() for k, t in m.state_dict().items()}
            else:
                bad += 5
                if bad >= PATIENCE:
                    break
    if state is not None:
        m.load_state_dict(state)
    m.eval()
    return m, ep + 1, best

## Scoring helpers

In [ ]:
def stats_fn(img, pcode, n_pat, W, center):
    def f(a, b):
        ok = np.isfinite(a) & np.isfinite(b)
        a, b, im, pc = a[ok].astype(np.float64), b[ok].astype(np.float64), img[ok], pcode[ok]
        if center:
            k = int(img.max()) + 1
            cnt = np.maximum(np.bincount(im, minlength=k), 1)
            a = a - (np.bincount(im, a, k) / cnt)[im]
            b = b - (np.bincount(im, b, k) / cnt)[im]
        else:
            a, b = a - a.mean(), b - b.mean()
        sxy, sxx, syy = (np.bincount(pc, a * b, n_pat), np.bincount(pc, a * a, n_pat), np.bincount(pc, b * b, n_pat))
        return float(sxy.sum() / np.sqrt(sxx.sum() * syy.sum())), (W @ sxy) / np.sqrt((W @ sxx) * (W @ syy))
    return f

def ci(v):
    v = np.asarray(v)[np.isfinite(v)]
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

def scorers(D, te, seed):
    tp, tcode = np.unique(D["patient"][te], return_inverse=True)
    Wt = np.random.default_rng(seed).multinomial(len(tp), np.full(len(tp), 1 / len(tp)), size=N_BOOT).astype(np.float64)
    return {"R4": stats_fn(D["img"][te], tcode, len(tp), Wt, True), "r": stats_fn(D["img"][te], tcode, len(tp), Wt, False)}, len(tp)

# Danenberg: frozen test patients (fixed across replicates)
D_TE = np.flatnonzero(DD["split"] == "test")
D_TRVA_PAT = np.unique(DD["patient"][DD["split"] != "test"])
SC_D, n_tpD = scorers(DD, D_TE, SEED + 13)
TRUTH_D = {m: {"cell mean": XD_in[D_TE, IN_SET.index(m)], "target map": TRUTH[m][D_TE]} for m in MARKERS}
print(f"Danenberg test: {len(D_TE):,} cells, {n_tpD} patients")

## Train the per-cell models (replicate-0 split and seeds of `train-test-grid`) and collect target-space predictions

In [ ]:
REP = 0
jp, jcode = np.unique(DJ["patient"], return_inverse=True)
u = np.random.default_rng(SEED + 11 + REP).random(len(jp))
J_SPLIT = np.where(u < 0.7, "train", np.where(u < 0.8, "val", "test"))[jcode]
uv = np.random.default_rng(SEED + 21 + REP).random(len(D_TRVA_PAT))
dval = set(D_TRVA_PAT[uv < 0.125])
D_SPLIT = np.where(DD["split"] == "test", "test",
                   np.where(pd.Series(DD["patient"]).isin(dval).values, "val", "train"))
assert np.array_equal(np.flatnonzero(D_SPLIT == "test"), D_TE)
J_TR, J_VA, D_TR = np.flatnonzero(J_SPLIT == "train"), np.flatnonzero(J_SPLIT == "val"), np.flatnonzero(D_SPLIT == "train")
print(f"Jackson train {len(J_TR):,} / val {len(J_VA):,} cells; Danenberg train {len(D_TR):,} cells")

# Jackson harmonisation statistics of the targets (the same numbers harm() used for H["Jackson"])
J_MED = np.nanmedian(XJ_in, 0)
_q05, _q95 = np.nanpercentile(XJ_in, [5, 95], 0)
J_SC = np.where(_q95 - _q05 > 1e-3, _q95 - _q05, np.nanstd(XJ_in, 0) + 1e-6)
_chk = np.random.default_rng(0).choice(len(XJ_in), 20_000, replace=False)
assert np.allclose((XJ_in[_chk] - J_MED) / J_SC, H["Jackson"][_chk, :XJ_in.shape[1]], atol=1e-4, equal_nan=True)

# Normalisation control: Danenberg inputs scaled with the Jackson statistics (no cohort-level adaptation)
JS = "Danenberg (Jackson-scaled)"
_la = DJ["log_area"].astype(np.float64)
LA_MED = np.nanmedian(_la)
_l05, _l95 = np.nanpercentile(_la, [5, 95])
LA_SC = _l95 - _l05 if _l95 - _l05 > 1e-3 else np.nanstd(_la) + 1e-6
assert np.allclose((_la[_chk] - LA_MED) / LA_SC, H["Jackson"][_chk, -1], atol=1e-4, equal_nan=True)
H[JS] = np.c_[(XD_in - J_MED) / J_SC, (DD["log_area"] - LA_MED) / LA_SC].astype(np.float32)
ADJ[JS] = ADJ["Danenberg"]
ensure({"Jackson", "Danenberg", JS})
print("Danenberg inputs, median over cells: own scaling", np.round(np.nanmedian(H["Danenberg"][:, :5], 0), 2),
      "| Jackson scaling", np.round(np.nanmedian(H[JS][:, :5], 0), 2))
CTRL = " [Jackson-scaled inputs]"

# Target normalisation of the image models (Jackson training percentiles, keyed by Jackson channel index)
j_idx = {m: i for i, m in enumerate(J_CHAN)}
CACHE = os.path.dirname(g("channel_preproc_residual.pkl")[0])
_, TNS = pickle.load(open(f"{CACHE}/channel_preproc_residual.pkl", "rb"))
print("target p01/p99:", {m: tuple(round(v, 3) for v in TNS[j_idx[m]]) for m in MARKERS})

def to_target(m, arc):
    p01, p99 = TNS[j_idx[m]]
    return np.clip((arc - p01) / (p99 - p01 + 1e-8), 0, 1).astype(np.float32)

HARMP, TSP, LOG = {}, {}, []      # harmonised predictions (Jackson-trained); target-space predictions (all)
Hs = H["Jackson"]
for m in MARKERS:
    k = IN_SET.index(m)
    cols = [i for i in range(Hs.shape[1]) if i != k]
    y_all = Hs[:, k]
    trm, vam = J_TR[np.isfinite(y_all[J_TR])], J_VA[np.isfinite(y_all[J_VA])]
    mu, sd = y_all[trm].mean(), y_all[trm].std() + 1e-6
    # Table, Jackson-trained (as in train-test-grid)
    fit = np.sort(np.random.default_rng(SEED + REP).choice(trm, min(N_FIT_FINAL, len(trm)), replace=False))
    t0 = time.time()
    mdl = lgb.LGBMRegressor(random_state=SEED + REP, **LGB).fit(Hs[fit][:, cols], y_all[fit])
    HARMP[("Table", m)] = mdl.predict(H["Danenberg"][D_TE][:, cols]).astype(np.float32)
    HARMP[("Table" + CTRL, m)] = mdl.predict(H[JS][D_TE][:, cols]).astype(np.float32)
    LOG.append(dict(model="Table", marker=m, epochs=np.nan, seconds=time.time() - t0, n_train=len(fit)))
    # MLP, GCN and GraphSAGE, Jackson-trained
    yt = torch.from_numpy(np.nan_to_num((y_all - mu) / sd)).to(dev)
    for name in ("MLP", "GCN", "GraphSAGE"):
        cls, akey = NETS[name]
        t0 = time.time()
        net, n_ep, best = train_net(cls, net_inputs(GR["Jackson"], akey, cols), yt, trm, vam, SEED + REP)
        with torch.no_grad():
            pdn = net(*net_inputs(GR["Danenberg"], akey, cols)).cpu().numpy() * sd + mu
            pdj = net(*net_inputs(GR[JS], akey, cols)).cpu().numpy() * sd + mu
        HARMP[(name, m)] = pdn[D_TE].astype(np.float32)
        HARMP[(name + CTRL, m)] = pdj[D_TE].astype(np.float32)
        LOG.append(dict(model=name, marker=m, epochs=n_ep, seconds=time.time() - t0, n_train=len(trm)))
        del net
        if dev == "cuda":
            torch.cuda.empty_cache()
    for name in ("Table", "MLP", "GCN", "GraphSAGE"):
        for nm in (name, name + CTRL):
            TSP[(nm, m)] = to_target(m, HARMP[(nm, m)] * J_SC[k] + J_MED[k])
    # Table, Danenberg-trained, directly on the target-space cell means of the train patients
    yD = TRUTH[m]
    trD = D_TR[np.isfinite(yD[D_TR])]
    fitD = np.sort(np.random.default_rng(SEED + REP).choice(trD, min(N_FIT_FINAL, len(trD)), replace=False))
    t0 = time.time()
    mdlD = lgb.LGBMRegressor(random_state=SEED + REP, **LGB).fit(H["Danenberg"][fitD][:, cols], yD[fitD])
    TSP[("Table (Danenberg-trained)", m)] = np.clip(mdlD.predict(H["Danenberg"][D_TE][:, cols]), 0, 1).astype(np.float32)
    LOG.append(dict(model="Table (Danenberg-trained)", marker=m, epochs=np.nan, seconds=time.time() - t0, n_train=len(fitD)))
    # existing image-model predictions and the painted-truth reference
    for a in ARMS:
        TSP[(f"ResNet-UNet {a}", m)] = RESNET[(a, m)][D_TE]
    TSP[("Painted true cell means", m)] = TRUTH[m][D_TE]
    tick(f"trained {m}")
pd.DataFrame(LOG).to_csv(f"{OUT}/training_log.csv", index=False)
MODEL_NAMES = [f"ResNet-UNet {a}" for a in ARMS] + ["Table", "MLP", "GCN", "GraphSAGE", "Table (Danenberg-trained)",
                                                    "Painted true cell means"] +               [n + CTRL for n in ("Table", "MLP", "GCN", "GraphSAGE")]

## Cell pixel counts; save per-cell predictions of the test cells

In [ ]:
def cell_counts(mask_path, obj):
    m = tifffile.imread(mask_path).astype(np.int64)
    lut = np.full(max(int(m.max()), int(obj.max())) + 1, -1, np.int64)
    lut[obj] = np.arange(len(obj))
    loc = lut[m.ravel()]
    return np.bincount(loc[loc >= 0], minlength=len(obj)).astype(np.int32), int(m.size)

_cc = Parallel(n_jobs=os.cpu_count())(delayed(cell_counts)(D_MASK[s], OBJ[a:b]) for s, a, b in zip(IMGS, ST[:-1], ST[1:]))
NCELL = np.concatenate([c for c, _ in _cc])
NPIX = np.array([n for _, n in _cc], np.int64)
assert len(NCELL) == N
tick("cell pixel counts")

IMG_TE = DD["img"][D_TE]
TEST_IMGS = np.unique(IMG_TE)
np.savez_compressed(f"{OUT}/percell_predictions_test.npz",
                    image=np.asarray(IMGS)[IMG_TE], object_number=OBJ[D_TE], patient=DD["patient"][D_TE],
                    n_pixels=NCELL[D_TE],
                    **{f"truth__{m}": TRUTH[m][D_TE] for m in MARKERS},
                    **{f"pred__{M}__{m}": TSP[(M, m)] for M in MODEL_NAMES for m in MARKERS},
                    **{f"harm__{M}__{m}": HARMP[(M, m)] for M in ("Table", "MLP", "GCN", "GraphSAGE") for m in MARKERS})
print(f"test: {len(D_TE):,} cells in {len(TEST_IMGS)} images")

## Per-image pixel sums for every model (painted per-cell maps; ResNet-UNet from `kill-test`)

In [ ]:
PIX = pd.read_csv(g("pixel_sums_per_image.csv")[0])
TRUTH_SUMS = PIX[PIX.arm == ARMS[0]].set_index(["marker", "image"])[["n", "Sg", "Sgg"]]
name_te = np.asarray(IMGS)[TEST_IMGS]
assert (TRUTH_SUMS.loc[MARKERS[0]].reindex(name_te).n.values == NPIX[TEST_IMGS]).all(), "pixel counts differ from kill-test"

K_IMG = len(IMGS)
nc = NCELL[D_TE].astype(np.float64)
rows = []
for m in MARKERS:
    gs = TRUTH_SUMS.loc[m].reindex(name_te)
    gc = np.nan_to_num(TRUTH[m][D_TE].astype(np.float64))
    for M in MODEL_NAMES:
        if M.startswith("ResNet-UNet"):
            arm = M.split()[-1]
            ps = PIX[(PIX.arm == arm) & (PIX.marker == m)].set_index("image").reindex(name_te)
            Sp, Spp, Sgp = ps.Sp.values, ps.Spp.values, ps.Sgp.values
        else:
            p = np.nan_to_num(TSP[(M, m)].astype(np.float64))
            Sp = np.bincount(IMG_TE, nc * p, K_IMG)[TEST_IMGS]
            Spp = np.bincount(IMG_TE, nc * p * p, K_IMG)[TEST_IMGS]
            Sgp = np.bincount(IMG_TE, nc * gc * p, K_IMG)[TEST_IMGS]
        rows.append(pd.DataFrame(dict(model=M, marker=m, image=name_te, n=gs.n.values, Sg=gs.Sg.values,
                                      Sgg=gs.Sgg.values, Sp=Sp, Spp=Spp, Sgp=Sgp)))
SUMS_TE = pd.concat(rows, ignore_index=True)
SUMS_TE.to_csv(f"{OUT}/pixel_sums_test.csv", index=False)
tick("per-image pixel sums")

def r_from(n, Sg, Sp, Sgg, Spp, Sgp):
    with np.errstate(invalid="ignore", divide="ignore"):
        return (Sgp - Sg * Sp / n) / np.sqrt((Sgg - Sg ** 2 / n) * (Spp - Sp ** 2 / n))

## Check: direct pixel r on 2 images equals the sums-based r

In [ ]:
CHECKS = []

@guard("direct pixel check")
def _():
    cand = [i for i in TEST_IMGS if ST[i + 1] - ST[i] > 500][:2]
    for i in cand:
        s = IMGS[i]
        im = tifffile.imread(D_IMG[s])
        if im.ndim == 3 and im.shape[0] == len(D_CHAN) and im.shape[-1] != len(D_CHAN):
            im = np.moveaxis(im, 0, -1)
        mask = tifffile.imread(D_MASK[s]).astype(np.int64)
        a, b = ST[i], ST[i + 1]
        lut = np.full(max(int(mask.max()), int(OBJ[a:b].max())) + 1, -1, np.int64)
        lut[OBJ[a:b]] = np.arange(b - a)
        loc = lut[mask]
        sel = np.searchsorted(D_TE, np.arange(a, b))                     # cell rows -> index in the test arrays
        assert (D_TE[sel] == np.arange(a, b)).all()
        for m in ("pan_cytokeratin", "cd3"):
            p01, p99 = TNS[j_idx[m]]
            G = np.clip((np.arcsinh(im[..., dix[m]].astype(np.float32) / COF) - p01) / (p99 - p01 + 1e-8), 0., 1.)
            for M in ("Table", "Table (Danenberg-trained)"):
                pc = np.nan_to_num(TSP[(M, m)][sel])
                Pm = np.where(loc >= 0, pc[np.maximum(loc, 0)], 0.0)
                r_dir = float(np.corrcoef(G.ravel().astype(np.float64), Pm.ravel())[0, 1])
                row = SUMS_TE[(SUMS_TE.model == M) & (SUMS_TE.marker == m) & (SUMS_TE.image == s)].iloc[0]
                r_sum = float(r_from(row.n, row.Sg, row.Sp, row.Sgg, row.Spp, row.Sgp))
                CHECKS.append(dict(image=s, marker=m, model=M, r_direct=r_dir, r_sums=r_sum,
                                   Sg_direct=float(G.sum(dtype=np.float64)), Sg_killtest=float(row.Sg)))
    ck = pd.DataFrame(CHECKS)
    ck.to_csv(f"{OUT}/direct_pixel_check.csv", index=False)
    print(ck.round(5).to_string(index=False))
    print("max |r_direct - r_sums| =", float((ck.r_direct - ck.r_sums).abs().max()))

In [ ]:
PATCH = 32                                        # 32 µm patches (1 µm/px), non-overlapping, partial edge tiles dropped
PATCH_CHECK = []

def patch_means(A):
    h, w = (A.shape[0] // PATCH) * PATCH, (A.shape[1] // PATCH) * PATCH
    return A[:h, :w].reshape(h // PATCH, PATCH, w // PATCH, PATCH).mean((1, 3)).ravel()

def patch_row(MODEL, m, s, G2, P2, T2=None):
    """Per-image sums over 32 µm patch means (truth G2, raw prediction P2, target-scale prediction T2)."""
    g, p = patch_means(G2), patch_means(P2)
    row = dict(model=MODEL, marker=m, image=s, n=g.size, Sg=g.sum(), Sgg=g @ g, Sp=p.sum(), Spp=p @ p, Sgp=g @ p)
    if T2 is not None:
        t = patch_means(T2)
        row.update(Sp_t=t.sum(), Spp_t=t @ t, Sgp_t=g @ t)
    if len(PATCH_CHECK) < 4 and g.size > 2 and g.std() > 0 and p.std() > 0:   # sums must give the direct patch r
        PATCH_CHECK.append(abs(float(np.corrcoef(g, p)[0, 1]) - float(r_from(*(row[c] for c in ["n", "Sg", "Sp", "Sgg", "Spp", "Sgp"])))))
    return row

# Per-cell models and the oracle, painted into the masks (background 0) in target space, as for the pixel sums.
# Their predictions are already in target space, so the target-scale sums (_t) equal the raw ones.
@guard("patch sums")
def _():
    rows = []
    for j, i in enumerate(TEST_IMGS):
        s = IMGS[i]
        im = tifffile.imread(D_IMG[s])
        if im.ndim == 3 and im.shape[0] == len(D_CHAN) and im.shape[-1] != len(D_CHAN):
            im = np.moveaxis(im, 0, -1)
        mask = tifffile.imread(D_MASK[s]).astype(np.int64)
        a, b = ST[i], ST[i + 1]
        lut = np.full(max(int(mask.max()), int(OBJ[a:b].max())) + 1, -1, np.int64)
        lut[OBJ[a:b]] = np.arange(b - a)
        loc = lut[mask]
        sel = np.searchsorted(D_TE, np.arange(a, b))
        assert (D_TE[sel] == np.arange(a, b)).all()
        for m in MARKERS:
            p01, p99 = TNS[j_idx[m]]
            G = np.clip((np.arcsinh(im[..., dix[m]].astype(np.float32) / COF) - p01) / (p99 - p01 + 1e-8), 0., 1.).astype(np.float64)
            if j < 3:   # the truth map must give kill-test's pixel sums
                ts = TRUTH_SUMS.loc[(m, s)]
                assert abs(G.sum() - ts.Sg) <= 1e-3 * max(1.0, abs(ts.Sg)) and G.size == ts.n, (s, m, G.sum(), ts.Sg)
            for M in MODEL_NAMES:
                if M.startswith("ResNet-UNet"):          # image model: patch sums come from unet-patch-sums
                    continue
                pc = np.nan_to_num(TSP[(M, m)][sel].astype(np.float64))
                Pm = np.where(loc >= 0, pc[np.maximum(loc, 0)], 0.0)
                rows.append(patch_row(M, m, s, G, Pm, Pm))
    pd.DataFrame(rows).to_csv(f"{OUT}/patch_sums_test.csv", index=False)
    assert PATCH_CHECK and max(PATCH_CHECK) < 1e-5, PATCH_CHECK
    print(f"patch sums: {len(rows)} rows, {len(TEST_IMGS)} test images; max |r_direct - r_sums| = {max(PATCH_CHECK):.2e}")
tick("patch sums")


## Metrics: (a) pooled pixel r, (b) per-image pixel r, (c) R4, with patient-bootstrap CIs

In [ ]:
tp, tcode = np.unique(DD["patient"][D_TE], return_inverse=True)
Wt = np.random.default_rng(SEED + 13).multinomial(len(tp), np.full(len(tp), 1 / len(tp)), size=N_BOOT).astype(np.float64)
pat_of_img = pd.Series(tcode, index=IMG_TE).groupby(level=0).first().reindex(TEST_IMGS).values
img_pat = pd.Series(pat_of_img, index=name_te)
cols6 = ["n", "Sg", "Sp", "Sgg", "Spp", "Sgp"]
EXCL = "estrogen_receptor_alpha"
BOOT, ROWS = {}, []

def record(M, metric, m, pt, bv, **extra):
    BOOT[(M, metric, m)] = (pt, bv)
    lo, hi = ci(bv)
    ROWS.append(dict(model=M, metric=metric, marker=m, value=pt, lo=lo, hi=hi, **extra))

for (M, m), df in SUMS_TE.groupby(["model", "marker"], sort=False):
    pc = img_pat.reindex(df.image).values.astype(np.int64)
    # (a) pooled pixel r
    per_pat = np.stack([np.bincount(pc, df[c].values, len(tp)) for c in cols6])
    record(M, "pooled pixel r", m, float(r_from(*per_pat.sum(1))), r_from(*(Wt @ per_pat.T).T))
    # (b) per-image pixel r, mean over images
    ri = r_from(*(df[c].values for c in cols6))
    ok = np.isfinite(ri)
    sr = np.bincount(pc[ok], ri[ok], len(tp))
    cn = np.bincount(pc[ok], minlength=len(tp)).astype(np.float64)
    record(M, "per-image pixel r", m, float(sr.sum() / cn.sum()), (Wt @ sr) / (Wt @ cn), n_images=int(ok.sum()))
    # (c) R4 on the same per-cell values
    pt, bv = SC_D["R4"](TRUTH[m][D_TE], TSP[(M, m)])
    record(M, "R4", m, pt, bv)

# R4 of the raw (harmonised, unclipped) predictions, to compare with train-test-grid replicate 0
for (M, m), v in HARMP.items():
    pt, bv = SC_D["R4"](TRUTH[m][D_TE], v)
    record(M + " [harmonised]", "R4", m, pt, bv)

PM = pd.DataFrame(ROWS)
PM.to_csv(f"{OUT}/metrics_per_marker.csv", index=False)

MED = []
for (M, metric), df in PM[PM.marker != EXCL].groupby(["model", "metric"], sort=False):
    ms = list(df.marker)
    pt = float(np.median([BOOT[(M, metric, x)][0] for x in ms]))
    bv = np.median(np.stack([BOOT[(M, metric, x)][1] for x in ms]), 0)
    lo, hi = ci(bv)
    MED.append(dict(model=M, metric=metric, value=pt, lo=lo, hi=hi, n_markers=len(ms)))
MED = pd.DataFrame(MED)
MED.to_csv(f"{OUT}/metrics_median.csv", index=False)
print("median over 10 markers (ERα excluded), Danenberg test patients:")
print(MED.pivot_table(index="model", columns="metric", values="value").round(3).to_string())
tick("metrics done")

In [ ]:
json.dump({"failed_blocks": FAILED, "minutes": (time.time() - T0) / 60}, open(f"{OUT}/run_info.json", "w"))
print("failed blocks:", FAILED if FAILED else "none")